# 🛡️ VoiceGuard — Real-Time AI Voice Clone Detection
### Setup & Model Loading

Run all cells top to bottom (`Runtime → Run all`). This installs dependencies, downloads the pretrained model, defines the detection pipeline, and launches the live demo app.

**Model:** Dhwani Multilingual Deepfake Audio Detection Model (Wav2Vec2 XLS-R + AASIST, ONNX) — trained on real & AI-generated speech across English, Hindi, Tamil, Telugu, and Malayalam, with telephony/transmission-artifact augmentation for real-world phone-call robustness.

## 1. Install Dependencies

In [ ]:
!pip install -q onnxruntime huggingface_hub librosa soundfile gradio
!apt-get -qq install -y ffmpeg

## 2. Load the Pretrained Model

Downloads model weights from Hugging Face and starts an ONNX Runtime inference session.

In [ ]:
from huggingface_hub import hf_hub_download
import onnxruntime as ort

MODEL_REPO = "ayush2635/Dhwani-Multilingual-Deepfake-Audio-Detection-Model"
MODEL_FILE = "best_model.onnx"

model_path = hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILE)
session = ort.InferenceSession(model_path)

TARGET_SR = 16000      # sample rate this model expects
MAX_LEN = 48000        # exactly 3 seconds at 16kHz

print("Model loaded successfully.")

## 3. Preprocessing Pipeline

Standardizes any uploaded audio file (mp3, wav, m4a) so it feeds into the model reliably:
resamples to 16kHz, converts to mono, trims silence, and guards against very short clips.

In [ ]:
import librosa
import numpy as np

def preprocess_audio(file_path, target_sr=TARGET_SR):
    """Loads any audio file, converts to mono, resamples to 16kHz, trims silence."""
    audio, sr = librosa.load(file_path, sr=target_sr, mono=True)
    audio, _ = librosa.effects.trim(audio, top_db=20)

    min_samples = target_sr  # 1 second minimum
    if len(audio) < min_samples:
        audio = np.pad(audio, (0, min_samples - len(audio)))

    return audio

## 4. Classification Engine

Wires preprocessing into the model. Returns a clean `{label, confidence, raw_scores}` dict.

In [ ]:
def classify_audio(file_path):
    """
    Takes an audio file path, returns:
    {'label': 'real' or 'spoof', 'confidence': float, 'raw_scores': {...}}
    """
    audio = preprocess_audio(file_path)

    # This model expects exactly 3 seconds (48000 samples) — pad or truncate
    if len(audio) > MAX_LEN:
        audio = audio[:MAX_LEN]
    else:
        audio = np.pad(audio, (0, MAX_LEN - len(audio)), mode='constant')

    # Normalize (zero mean, unit variance) — required by this model
    audio = (audio - np.mean(audio)) / np.sqrt(np.var(audio) + 1e-5)
    audio = audio.astype(np.float32).reshape(1, MAX_LEN)

    input_name = session.get_inputs()[0].name
    logits = session.run(None, {input_name: audio})[0]

    probs = np.exp(logits) / np.sum(np.exp(logits), axis=1, keepdims=True)
    probs = probs[0]

    labels = {0: "real", 1: "spoof"}
    pred_id = int(np.argmax(probs))

    return {
        "label": labels[pred_id],
        "confidence": float(probs[pred_id]),
        "raw_scores": {labels[0]: float(probs[0]), labels[1]: float(probs[1])}
    }

## 5. Explainability Layer

Translates the raw probability score into a plain-language verdict, confidence tier, and advice — so the result is useful to a non-technical user, not a black box.

In [ ]:
def explain_result(result):
    """Takes classify_audio() output, returns a user-friendly explanation dict."""
    label = result["label"]
    confidence = result["confidence"]
    conf_pct = confidence * 100

    if confidence > 0.90:
        tier = "High confidence"
        advice = "This is very likely correct." if label == "real" else "Strongly recommend verifying through another channel before acting."
    elif confidence > 0.60:
        tier = "Moderate confidence"
        advice = "Proceed with caution — consider verifying independently."
    else:
        tier = "Uncertain"
        advice = "Please verify through another channel (call back on a known number)."

    if label == "spoof":
        verdict = "⚠️ Likely AI-Generated / Cloned Voice"
        signal_note = ("AI voice synthesis often leaves subtle traces in pitch "
                        "consistency, spectral texture, and micro-pauses that "
                        "differ from natural human speech patterns.")
    else:
        verdict = "✅ Likely Real Human Voice"
        signal_note = ("Natural variation in pitch, breathing, and background "
                        "acoustics is consistent with genuine human speech.")

    return {
        "verdict": verdict,
        "confidence_pct": round(conf_pct, 1),
        "tier": tier,
        "advice": advice,
        "signal_note": signal_note
    }

## 6. Demo Dataset Evaluation *(optional — run to reproduce our test results)*

Runs the full pipeline against our curated 10-clip dataset (5 real, 5 AI-cloned scam scripts spanning family-emergency and bank/institution-impersonation scenarios). Requires the files to be uploaded to `/content/` first.

In [ ]:
import os

test_files = {
    "Loan_cloned": "spoof", "Creditcard_cloned": "spoof", "Incometax_cloned": "spoof",
    "Lostphone_cloned": "spoof", "Bank_kyc_cloned": "spoof",
    "Kyc_update_real": "real", "Incometax_real": "real", "Creditcard_real": "real",
    "Loan_real": "real", "Lostphone_real": "real"
}

print(f"{'File':<20} {'True':<8} {'Predicted':<10} {'Conf %':<8} {'Correct?'}")
for name, true_label in test_files.items():
    for ext in [".mp3", ".m4a", ".wav"]:
        path = f"/content/{name}{ext}"
        if os.path.exists(path):
            result = classify_audio(path)
            correct = "✅" if result["label"] == true_label else "❌"
            print(f"{name:<20} {true_label:<8} {result['label']:<10} {result['confidence']*100:.1f}%   {correct}")
            break

## 7. VoiceGuard App (Gradio UI)

The live, judge-facing interface. Upload or record a clip, click Analyze, get an instant color-coded verdict with confidence score and explanation.

In [ ]:
import gradio as gr

def analyze_audio(audio_file):
    if audio_file is None:
        return "### Please upload or record an audio clip.", "", "", gr.update(visible=False)

    result = classify_audio(audio_file)
    explanation = explain_result(result)

    is_spoof = (result["label"] == "spoof")
    color = "#e74c3c" if is_spoof else "#2ecc71"  # red for spoof, green for real

    verdict_md = f"""
    <div style="padding: 16px; border-radius: 10px; background-color: {color}22; border: 2px solid {color};">
        <h2 style="color: {color}; margin: 0;">{explanation['verdict']}</h2>
        <p style="margin: 4px 0 0 0; font-size: 15px;">{explanation['tier']} — {explanation['confidence_pct']}%</p>
    </div>
    """

    advice_md = f"**What to do:** {explanation['advice']}"
    detail_md = f"*{explanation['signal_note']}*"

    return verdict_md, advice_md, detail_md, gr.update(value=explanation['confidence_pct'], visible=True)

with gr.Blocks(title="VoiceGuard") as demo:
    gr.Markdown("# 🛡️ VoiceGuard")
    gr.Markdown("**Detect AI voice clones in real time — before you send money.**")

    audio_input = gr.Audio(sources=["upload", "microphone"], type="filepath", label="Upload or Record Suspicious Audio")
    analyze_btn = gr.Button("🔍 Analyze", variant="primary", size="lg")

    verdict_output = gr.HTML()
    confidence_bar = gr.Slider(minimum=0, maximum=100, label="Confidence Score", interactive=False, visible=False)
    advice_output = gr.Markdown()
    detail_output = gr.Markdown()

    with gr.Accordion("ℹ️ How this works", open=False):
        gr.Markdown("""
        VoiceGuard uses a Wav2Vec2 + AASIST deep learning model trained to detect
        synthetic speech artifacts — subtle irregularities in pitch, spectral
        texture, and timing that AI voice generators tend to leave behind,
        even when the result sounds convincing to the human ear.

        Trained on multilingual data including telephony-condition audio,
        so it's built for real phone-call scenarios, not just studio clips.

        **Note:** This is a decision-support tool, not a guarantee. Always
        verify unusual money requests through a separate, trusted channel —
        call the person back on a number you already know.
        """)

    analyze_btn.click(
        fn=analyze_audio,
        inputs=audio_input,
        outputs=[verdict_output, advice_output, detail_output, confidence_bar]
    )

demo.launch(share=True)